# Complete native flexible quantum phase estimation

Retain the source two-qubit Hamiltonian, duplicate ordered ZZ terms, fresh
NumPy eigendecomposition and eigenvector index 2, five phase bits, p0=4,
r0=4 and gamma=1.5. The approximate family applies first-order product formulas
with repetitions (4,4,4,12,32) at powers (1,2,4,8,16). Exact comparison uses the
same data register after approximate QPE, then a controlled-SWAP overlap test
between the two phase registers. No reset or independent-copy substitution is
made. Dropped registers remain the private environment in full-state checks.

QPE and scalar gates operate on encoded storage while QNum retains exact binary
scale for numeric readout. Raw scalar gate access does not grant unsigned sliced
arithmetic or change the numeric format. SWAP is expressed with three CX gates;
its scalar control reuses existing controlled gates. Exact matrix exponentiation
uses NumPy eigh of this Hermitian source matrix instead of adding SciPy.

The source does not specify the default sample count. This local counterpart
explicitly configures 1024 independent draws per experiment and records seeds.
Counts are observations, not copied historical results. No cloud job, backend
change, hardware, optimality or speedup claim is included.


In [ ]:
from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.tutorials.advanced_tutorials.high_level_modeling_flexible_qpe.overview.ansatz import (
    HAMILTONIAN,
    build_program,
    exact_matrix,
    repetition_table,
    source_eigenpair,
    source_matrix,
)
from examples.synthesis.tutorials.advanced_tutorials.high_level_modeling_flexible_qpe.overview.plots import (
    plot_phase,
)
from examples.synthesis.tutorials.advanced_tutorials.high_level_modeling_flexible_qpe.overview.workflow import (
    verify_and_sample,
)

print("Public native qfunc and independent Circuit verification")


In [ ]:
print("Per-power family uses native qpe(lambda power: helper(...), phase)")
print("Native QPE retains H, coherent controlled family members and inverse Fourier")


In [ ]:
p0 = 4
r0 = 4
gamma = 1.5
schedule = tuple((power, repetition_table()[power]) for power in (1, 2, 4, 8, 16))
assert schedule == ((1, 4), (2, 4), (4, 4), (8, 12), (16, 32))
print("Original first-order evolution schedule:", schedule)


In [ ]:
print("Original Hamiltonian:", HAMILTONIAN)
print("Independent fresh dense matrix:", source_matrix())


In [ ]:
eigenvalue, amplitudes = source_eigenpair()
print("Chosen eigenvector (index 2):", amplitudes)
print("Exact eigenvalue to estimate:", eigenvalue)


In [ ]:
phase_bits = 5
estimate_program = build_program()
print("Captured native estimation program, with five fractional phase bits")


In [ ]:
print("Actual estimation resources:", estimate_program.report["resources"])
print("Named estimation ports:", estimate_program.registers)


In [ ]:
estimate = verify_and_sample(estimate_program, seed=4100, shots=1024)
print("Absolute full-state error:", estimate.state_error)
print("Named numeric error:", estimate.named_error)


In [ ]:
phase_counts = {row.word / 32: row.count for row in estimate.outcomes}
print(phase_counts)


In [ ]:
output_directory = Path("artifacts/development/flexible_qpe")
plot_phase(estimate, output_directory)
figure_path = output_directory / "phase_estimate.png"
display(Image(filename=str(figure_path)))
print("Exact eigenvalue:", estimate.exact_eigenvalue)
print("Observed most likely phase:", estimate.most_likely_phase)


In [ ]:
print("Exact coherent family:", "matrix power via native static helper")
print("Fresh Hermitian exponential:", exact_matrix())


In [ ]:
comparison_program = build_program(comparison=True)
print("Same data register is retained across approximate and exact QPE")
print("Scalar controlled three-CX exchanges compare all five phase pairs")


In [ ]:
print("Actual comparison resources:", comparison_program.report["resources"])
print("Named output:", comparison_program.registers)


In [ ]:
comparison = verify_and_sample(
    comparison_program, comparison=True, seed=4101, shots=1024
)
print("Absolute thirteen-qubit state error:", comparison.state_error)
print("Named comparison error:", comparison.named_error)


In [ ]:
test_counts = {str(row.word): row.count for row in comparison.outcomes}
print(test_counts)


In [ ]:
print("Observed overlap statistic:", comparison.observed_overlap)
print("Exact finite-QPE overlap statistic:", comparison.exact_overlap)
print(
    "Source stored observed overlap 0.9609375 is historical, not a required sampled value"
)
print(
    "Shared data can remain entangled: this statistic is not asserted to be the fidelity of independent pure phase states"
)
